In [0]:
%run ../utils/adls_auth

In [0]:
# Disable deletion vectors for Synapse compatibility
spark.conf.set("spark.databricks.delta.properties.defaults.enableDeletionVectors", "false")

In [0]:
from pyspark.sql.functions import sha2, col
from delta.tables import DeltaTable


GOLD_PATH = "abfss://gold@stdatalakenyctaxi.dfs.core.windows.net/dim_vendor"


In [0]:

vendor_df = spark.createDataFrame(
    [
        (1, "Creative Mobile Technologies, LLC"),
        (2, "Curb Mobility, LLC"),
        (6, "Unknown / New Vendor"),  
    ],
    schema="vendor_id INT, vendor_name STRING",
).withColumn("vendor_key", sha2(col("vendor_id").cast("string"), 256))

if not DeltaTable.isDeltaTable(spark, GOLD_PATH):
    vendor_df.write.format("delta").mode("overwrite").save(GOLD_PATH)
else:
    dim_table = DeltaTable.forPath(spark, GOLD_PATH)
    (dim_table.alias("target")
        .merge(vendor_df.alias("source"), "target.vendor_id = source.vendor_id")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())

print(f"dim_vendor built: {vendor_df.count()} rows.")